In [1]:
#import shutil
#shutil.rmtree("Feature_Targets")

In [2]:
#!pip install --user duckdb

In [3]:
import pandas as pd
import numpy as np
import os
import time
from tqdm.notebook import tqdm
import duckdb
import glob
import re
from joblib import Parallel, delayed


import importlib
import utils.data_preprocessing as du

importlib.reload(du)

<module 'utils.data_preprocessing' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils/data_preprocessing.py'>

In [5]:
def process_date(date):
    con = duckdb.connect()
    con.execute("SET enable_progress_bar = false")
    
    fut_df = con.execute(f"""
        SELECT "Timestamp", "MicroPrice_tick-based_10_1s" AS MicroPrice
        FROM read_parquet('Price_Measures/{date}/FUT_DAX_Futures.parquet')
    """).df()

    fut = du.compute_feature_target_matrix(
        df=fut_df,
        ts_col='Timestamp',
        target_cols=[],
        feature_cols=['MicroPrice'],
        horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-10s', '-2s', '-1s', '-100ms'])
    fut = fut.sort_values("Timestamp").reset_index(drop=True)
    
    cols = ["Timestamp", "Timestamp_Europe/Berlin", "L1-BidPrice", "L1-AskPrice", "Side",
            "L1-QImb", "MidPrice", "MidPriceQW", "MidPriceCQW",
            "MicroPrice_tick-based_10_1s"]

    df = con.execute(f"""
        SELECT {", ".join(f'"{c}"' for c in cols)},
               "MicroPrice_tick-based_10_1s" AS MicroPrice,
               regexp_extract(filename, '[^/]+/([^/]+)\\.parquet$', 1) AS Symbol
        FROM read_parquet('Price_Measures/{date}/*.parquet', filename=true)
        WHERE regexp_extract(filename, '[^/]+/([^/]+)\\.parquet$', 1) != 'FUT_DAX_Futures'
    """).df()

    os.makedirs(f'Feature_Targets/{date}', exist_ok=True)
    
    for symbol_name, group in df.groupby("Symbol"):
        filtered = du.filter_trading_hours(group).copy()
        filtered['TransactionPrice'] = du.compute_transaction_price(filtered)
        featured = du.compute_feature_target_matrix(
            df=filtered,
            ts_col='Timestamp',
            target_cols=du.PRICE_MEASURES,
            feature_cols=['L1-QImb', 'MicroPrice'],
            horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-10s', '-2s', '-1s', '-100ms',
                      '100ms', '1s', '2s', '10s', '15s', '30s', '1m', '2.5m', '5m'])
        merged = pd.merge_asof(
            featured, fut,
            on="Timestamp",
            direction="backward",
            suffixes=("_LOB", "_FUT"))
        merged = merged.drop(columns=['TransactionPrice', 'MidPriceQW', 'MidPriceCQW',
                                      'MicroPrice_FUT', 'MicroPrice_LOB'])
        merged.to_parquet(f'Feature_Targets/{date}/{symbol_name}.parquet')

# Run
overall_start = time.time()
Parallel(n_jobs=2)(delayed(process_date)(date) for date in tqdm(du.SAMPLE_DATES[:8], desc="Dates"))
print(f"Overall completed in {time.time() - overall_start:.2f}s")

Dates:   0%|          | 0/8 [00:00<?, ?it/s]

ValueError: time data "2023-01-09 17:03:54+01:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 526534. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.